# 🧪 Thực Nghiệm Đo Đạc & So Sánh Hệ Số Lipschitz Thực Nghiệm
### **Vanilla LiDAR vs. RS-LiDAR (Randomized Smoothing)** trên 2x GPU Tesla T4

Notebook này được thiết kế độc lập để kiểm chứng **Định lý Giảm Lipschitz** (Theorem 1 & Proposition 2 trong `LIPSCHITZ_REDUCTION_ANALYSIS.md`):
- **Định nghĩa Giải tích**: $L = \sup \frac{|R(x_1) - R(x_2)|}{\|x_1 - x_2\|_2}$. Mọi thương số sai phân trên cặp ảnh đều là một chặn dưới thực nghiệm hợp lệ ($L \ge L_{\max}$).
- **Tính đối xứng nghiêm ngặt**: Hàm smoothed reward $R_{\sigma_2}$ Monte Carlo được tính độc lập cho **CẢ ẢNH SẠCH $x_{\text{clean}}$ VÀ ẢNH BIẾN DẠNG $x_{\text{pert}}$**:
  $$R_{\sigma_2}(x) = \frac{1}{M} \sum_{m=1}^M R(x + \sigma_2 u_m), \quad u_m \sim \mathcal{N}(0, I)$$
  $$\text{Slope}_{\text{RS}} = \frac{|R_{\sigma_2}(x_{\text{clean}}) - R_{\sigma_2}(x_{\text{pert}})|}{\|x_{\text{clean}} - x_{\text{pert}}\|_2}$$
- **4 Mô hình Reward Đánh giá**: `ImageReward`, `CLIP-Score`, `HPS v2.1`, `Aesthetic Score`.
- **Chạy song song 2 GPU (`GPU T4 x2`)**: Tự động phân chia Shard 0 (GPU 0) và Shard 1 (GPU 1), stream log trực tiếp và tự động gộp kết quả.

## 1. Kiểm Tra Phần Cứng GPU & Cấu Hình Bộ Nhớ
Đảm bảo bạn đã chọn **GPU T4 x2** trong Session options bên phải giao diện Kaggle.

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Settings -> Accelerator -> GPU T4 x2.")

# Chống phân mảnh bộ nhớ VRAM trên GPU T4 16GB
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
!nvidia-smi

## 2. Đồng Bộ Mã Nguồn Repo & Cài Đặt Môi Trường
Cài đặt các gói phụ thuộc và chuẩn bị file từ vựng HPS v2.1 chống lỗi mạng.

In [ ]:
import os, shutil, glob, json, random, subprocess, threading, urllib.request

# 1. Đồng bộ repo RS-LiDAR
REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling" if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling") else REPO_DIR
os.chdir(WORKDIR)
%cd {WORKDIR}
print("📂 Thư mục làm việc:", os.getcwd())

# 2. Cài đặt các thư viện thiết yếu
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q image-reward || pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Chuẩn bị file vocab cho HPS v2.1
try:
    import hpsv2
    hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
    os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
    if not os.path.exists(hpsv2_vocab):
        urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
    print("✅ File vocab HPS v2.1 đã sẵn sàng!")
except Exception as e:
    print(f"Lưu ý khi chuẩn bị HPS vocab: {e}")

# 4. Tiện ích chạy song song 2 GPU hiển thị log trực tiếp
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    rc0 = p0.wait()
    rc1 = p1.wait()
    if rc0 == 0:
        print("✅ [GPU 0] Tiến trình hoàn thành thành công.")
    else:
        print(f"❌ [GPU 0] Tiến trình dừng với mã lỗi rc={rc0}!")
    if rc1 == 0:
        print("✅ [GPU 1] Tiến trình hoàn thành thành công.")
    else:
        print(f"❌ [GPU 1] Tiến trình dừng với mã lỗi rc={rc1}!")
    if rc0 != 0 or rc1 != 0:
        raise RuntimeError(f"Tiến trình GPU thất bại (GPU 0: {rc0}, GPU 1: {rc1})")

print("\n✅ Môi trường thực nghiệm đo Lipschitz đã sẵn sàng 100%!")

## 3. Cấu Hình Tham Số Thực Nghiệm
Tất cả tham số dòng lệnh được định nghĩa tập trung tại đây:
- **50 Prompts Phân Tầng** (`geneval_50_stratified.jsonl`): Đại diện đều cho 6 tasks của GenEval.
- **10 Ảnh / Prompt** ($K=10$): Tổng cộng 500 cặp mẫu ảnh ($x_{\text{clean}}, x_{\text{pert}}$).
- **$\sigma_1 = 0.1$**: Vi nhiễu tạo ảnh biến dạng (thước đo vi phân probe scale).
- **$\sigma_2 = 1.0$**: Bán kính làm mịn Sweet Spot tối ưu của RS-LiDAR.
- **$\sigma_2$ Sweep**: Quét dải $\sigma_2 \in \{0.0, 0.1, 0.25, 0.5, 1.0\}$ để vẽ đường cong suy giảm Lipschitz $\mathcal{O}(1/\sigma)$.
- **$M = 4$**: Số mẫu Monte Carlo xấp xỉ kỳ vọng cho cả ảnh sạch và ảnh biến dạng.

In [ ]:
# ==============================================================================
# ⚙️ KHU VỰC CẤU HÌNH THAM SỐ ĐO LIPSCHITZ (TÙY CHỈNH TẠI ĐÂY)
# ==============================================================================
NUM_PROMPTS = 50                     # 50 prompts phân tầng đại diện GenEval
NUM_PARTICLES = 10                  # 10 ảnh / prompt -> 500 cặp ảnh (1000 ảnh đánh giá)
SIGMA1 = 0.1                        # Độ lệch chuẩn vi nhiễu tạo x_pert
PERTURBATION_TYPE = "gaussian"      # Kiểu biến dạng: "gaussian" hoặc "blur"
SIGMA2 = 1.0                        # Bán kính làm mịn chính của RS-LiDAR (Sweet Spot)
SIGMA2_SWEEP = "0.0,0.1,0.25,0.5,1.0" # Dải quét sigma2 vẽ đường cong ablation
NUM_MC_SAMPLES = 4                  # M = 4 mẫu Monte Carlo tính kỳ vọng
OVERWRITE = False                   # Đặt True nếu muốn chạy lại từ đầu (False để Resume)
# ==============================================================================

PROMPT_FILE = "prompt_files/geneval_50_stratified.jsonl"
TEST_OUTPUT_DIR = "/kaggle/working/results/lipschitz_empirical"
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

print(f"⚙️ Cấu hình: {NUM_PROMPTS} prompts | {NUM_PARTICLES} hạt/prompt | Tổng: {NUM_PROMPTS * NUM_PARTICLES} cặp mẫu")
print(f"🔬 Nhiễu biến dạng sigma1 = {SIGMA1} ({PERTURBATION_TYPE}) | Bán kính làm mịn RS-LiDAR sigma2 = {SIGMA2}")
print(f"📈 Dải khảo sát sigma2: {{{SIGMA2_SWEEP}}} | Monte Carlo M = {NUM_MC_SAMPLES}")
print(f"📁 Thư mục lưu kết quả: {TEST_OUTPUT_DIR}")

## 4. Tiền Tải Trọng Số & Thực Thi Đo Lipschitz Song Song 2x GPU T4
Quá trình tiền tải model weights vào cache ổ đĩa để tránh xung đột đa tiến trình, sau đó khởi chạy Shard 0 (GPU 0) và Shard 1 (GPU 1) song song.

In [ ]:
# 1. Tự động kiểm tra & tự phục hồi thư mục làm việc chuẩn
if not os.path.exists("test_empirical_lipschitz.py"):
    for cand in ["/kaggle/working/RS-LiDAR/Diffusion-LiDAR-Sampling", "/kaggle/working/RS-LiDAR", "."]:
        if os.path.exists(os.path.join(cand, "test_empirical_lipschitz.py")):
            os.chdir(cand)
            print("📂 Đã chuyển đến thư mục làm việc:", os.getcwd())
            break

# 2. Dọn dẹp tiến trình cũ nếu có và giải phóng VRAM
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
!pkill -f "test_empirical_lipschitz.py" || true

# 3. Tiền tải trọng số Stable Diffusion 1.5, ImageReward, HPSv2, Aesthetic, CLIP
print("📥 Đang kiểm tra & tiền tải toàn bộ model weights chuẩn vào cache ổ đĩa...")
from huggingface_hub import snapshot_download
snapshot_download(
    "runwayml/stable-diffusion-v1-5",
    allow_patterns=["model_index.json", "*scheduler/*", "*tokenizer*/*", "*text_encoder*/*model.safetensors", "*unet*/*diffusion_pytorch_model.safetensors", "*vae*/*diffusion_pytorch_model.safetensors"],
    ignore_patterns=["*.ckpt", "*pruned*", "*non_ema*", "safety_checker/*", "*.bin", "*fp16*"]
)

# ImageReward weights
ir_path = os.path.expanduser("~/.cache/ImageReward/ImageReward.pt")
if not os.path.exists(ir_path):
    os.makedirs(os.path.dirname(ir_path), exist_ok=True)
    try:
        urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/ImageReward.pt", ir_path)
        urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/med_config.json", os.path.join(os.path.dirname(ir_path), "med_config.json"))
        print("✅ Đã tiền tải xong ImageReward weights!")
    except Exception as e:
        print(f"Lưu ý tải ImageReward: {e}")

# HPS v2.1 weights
hps_ckpt = os.path.expanduser("~/.cache/hpsv2/HPS_v2.1_compressed.pt")
if not os.path.exists(hps_ckpt) or os.path.getsize(hps_ckpt) < 1000000:
    os.makedirs(os.path.dirname(hps_ckpt), exist_ok=True)
    try:
        urllib.request.urlretrieve("https://huggingface.co/xswu/HPSv2/resolve/main/HPS_v2.1_compressed.pt", hps_ckpt)
        print("✅ Đã tải xong HPSv2.1!")
    except Exception as e:
        print(f"Lưu ý tải HPSv2: {e}")

# Aesthetic Score weights
as_path = "sac+logos+ava1-l14-linearMSE.pth"
if not os.path.exists(as_path) or os.path.getsize(as_path) < 100000:
    try:
        urllib.request.urlretrieve("https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac%2Blogos%2Bava1-l14-linearMSE.pth", as_path)
        print("✅ Đã tải xong Aesthetic Score weights!")
    except Exception as e:
        print(f"Lưu ý tải Aesthetic weights: {e}")

# OpenAI CLIP ViT-L/14 Weights
try:
    import clip
    _ = clip.load("ViT-L/14", device="cpu")
    print("✅ Trọng số OpenAI CLIP ViT-L/14 đã sẵn sàng trong cache!")
except Exception as e:
    print(f"Lưu ý nạp CLIP: {e}")

# 4. Khởi chạy đo Lipschitz song song 2 GPU
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
ov_flag = "--overwrite" if OVERWRITE else ""

print(f"\n🚀 Bắt đầu đo hệ số Lipschitz thực nghiệm trên {n_gpus} GPU...\n")

if n_gpus >= 2:
    print("⚡ [2 GPU] Đang chạy song song trên GPU 0 (Shard 0) và GPU 1 (Shard 1)...")
    cmd0 = f"""python -u test_empirical_lipschitz.py \
        --prompts_file "{PROMPT_FILE}" \
        --num_prompts {NUM_PROMPTS} \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --num_mc_samples {NUM_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 0 --gpu_id 0 {ov_flag}"""

    cmd1 = f"""python -u test_empirical_lipschitz.py \
        --prompts_file "{PROMPT_FILE}" \
        --num_prompts {NUM_PROMPTS} \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --num_mc_samples {NUM_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 1 --gpu_id 1 {ov_flag}"""
    run_commands_parallel(cmd0, cmd1)
else:
    print("⚡ [1 GPU] Đang chạy đo Lipschitz trên 1 GPU...")
    !python -u test_empirical_lipschitz.py \
        --prompts_file "{PROMPT_FILE}" \
        --num_prompts {NUM_PROMPTS} \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --num_mc_samples {NUM_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" {ov_flag}

print("\n✅ HOÀN THÀNH ĐO ĐẠC! Tiến hành gộp shard...")

## 5. Tự Động Gộp Kết Quả Shards & Tính Toán Thống Kê
Lệnh `--merge_shards` tự động tổng hợp toàn bộ 500 cặp mẫu từ Shard 0 và Shard 1, tính toán các chỉ số $L_{\max}, L_{\text{mean}}, L_{95\%}$, Reduction Ratio, và vẽ đồ thị.

In [ ]:
print("🔄 Đang gộp kết quả từ các shard checkpoints...")
!python -u test_empirical_lipschitz.py --merge_shards --output_dir "{TEST_OUTPUT_DIR}" --sigma2 {SIGMA2}
print("✅ Hoàn tất gộp dữ liệu và xuất báo cáo!")

## 6. Hiển Thị Bảng Kết Quả Chi Tiết & Bộ Đồ Thị 3-Panel
Hiển thị trực tiếp bảng tóm tắt hệ số Lipschitz thực nghiệm, bảng khảo sát $\sigma_2$ ablation, và các biểu đồ phân phối độ dốc chuẩn công bố quốc tế.

In [ ]:
import pandas as pd, glob, os
from IPython.display import display, Image, Markdown

# 1. Bảng Tổng Hợp Hệ Số Lipschitz (Vanilla vs. RS-LiDAR)
summary_csv = os.path.join(TEST_OUTPUT_DIR, "lipschitz_summary.csv")
if os.path.exists(summary_csv):
    df_summary = pd.read_csv(summary_csv)
    print("=" * 115)
    print("🏆 BẢNG TỔNG HỢP HỆ SỐ LIPSCHITZ THỰC NGHIỆM: VANILLA LIDAR VS. RS-LIDAR")
    print("=" * 115)
    display(df_summary)

# 2. Bảng Khảo Sát Bán Kính Làm Mịn Sigma2 (Ablation Table)
sweep_csv = os.path.join(TEST_OUTPUT_DIR, "lipschitz_sigma_ablation.csv")
if os.path.exists(sweep_csv):
    df_sweep = pd.read_csv(sweep_csv)
    print("\n" + "=" * 115)
    print(f"📊 BẢNG KHẢO SÁT BÁN KÍNH LÀM MỊN SIGMA2 (σ ∈ {{{SIGMA2_SWEEP}}}):")
    print("=" * 115)
    display(df_sweep)

# 3. Hiển thị Đồ thị 3-Panel và Đồ thị Ablation
plot_files = sorted(glob.glob(f"{TEST_OUTPUT_DIR}/*.png"))
for pf in plot_files:
    print(f"\n📈 Đồ thị: {os.path.basename(pf)}")
    display(Image(filename=pf))

## 7. Đóng Gói Toàn Bộ Kết Quả (1-Click Download)
Nén toàn bộ các file CSV, JSON metrics và hình vẽ PNG vào `/kaggle/working/results_lipschitz.zip` để bạn có thể tải về trực tiếp từ tab **Output** bên phải Kaggle.

In [ ]:
zip_path = "/kaggle/working/results_lipschitz.zip"
print(f"📦 Đang nén toàn bộ kết quả đo Lipschitz vào {zip_path}...")

!zip -r -q {zip_path} {TEST_OUTPUT_DIR}

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    print(f"📁 Đường dẫn file zip: {zip_path}")
    print("👉 Bạn có thể tải file này trực tiếp từ cột 'Output' bên phải giao diện Kaggle!")
else:
    print("⚠️ Không tìm thấy file zip được tạo.")